# Long Baseline Study of Mira Variable Stars: Part Two
## Glass Plate Analysis
The majority of this notebook is taken from Emily's New Plate Analysis notebook, with adjustments made to calculate the magnitude in the end. Check each cell for multiple hashtags that indicate where you need to input your own info. Skip to Step five after running the imports and path if you already have a solved WCS file.

#### Step One
Take an inner scan of your glass plate and transfer the .tif file.

#### Step Two
Convert from .tif to .fits and provide header information from the finding aids.

In [ ]:
#Start with yor imports!
import cv2
import numpy as np
from astropy.io import fits
from astroquery.gaia import Gaia
from astropy.coordinates.sky_coordinate import SkyCoord
from astropy.wcs import WCS
import sep
sep.set_extract_pixstack(5000000)
sep.set_sub_object_limit(1000000)

from astropy.table import Table, hstack
from scipy.optimize import curve_fit
import csv
import math 
from scipy.stats import mode as spmode
import statistics

import matplotlib.pyplot as plt
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground
from photutils.background import *
from photutils.datasets import make_100gaussians_image
from astropy.stats import biweight_location
from astropy.stats import mad_std
from astropy.stats import sigma_clipped_stats

import astropy.table
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from astropy.wcs import WCS
from astropy.table import Table, hstack

import os
from astropy.io import ascii
import glob
from glob import glob

from astroquery.astrometry_net import AstrometryNet

# sq func used in analysis 
def sq(x,a,b):
    return(a*x**(b/2))
import pandas as pd

from datetime import datetime

mean = np.mean
std = np.std

In [ ]:
# defining functions....do not alter these please


# function to convert tif files into fits files
def tif2fits_RGB(file, write_header = False):  
    tif = cv2.imread(file,cv2.IMREAD_UNCHANGED)
    dat = tif.astype(np.float32)
    g,b,r = cv2.split(dat)
    data = (b/3) + (g/3) + (r/3)
    newfname = file[:-4]+'.fits'
    print(f"\n\033[1mCreating file... {os.path.basename(newfname)}\033[0m")
    if write_header == True: 
        new_header = create_header()
    elif write_header == False:
        new_header = fits.Header()
    fits.writeto(newfname, data = -data+np.max(data), header = new_header, overwrite = True)
    os.chmod(newfname, 0o777)
    return newfname



# function to create a fits header from finding aid information
def create_header():

    ############ DO NOT CHANGE THE FOLLOWING PATH ###############
    path = '/raid6/yerkes/Student_Workshops/YAPSS_v3.0.0/Finding Aids/Matching/'
    
    files = glob(path + '*.csv')
    
    for w, file in enumerate(files):
        print('\t',w, os.path.basename(file))
    
    
    # indicate which finding aid you would like to alter
    
    input_file = files[int(input('\nWhat is the FILE INDEX of your finding aid?'))]
    print('\tYou chose the following file: ' + os.path.basename(input_file))
    
    df = pd.read_csv(input_file, header = 0)
    t0 = Table.from_pandas(df)

    correct_plate = False
    while correct_plate == False: 
        table_mask = np.where(t0['Rcd'] == int(input('\nWhat is the record number (Rcd) of your plate? (NOT THE PLATE NUMBER) ')))[0][0]
        t1 = t0[table_mask]
        print(f" \t You chose the following plate:  {(t1['Series'])}  {(t1['Plate no.'])}")
    
        correct = input('\tIs this the correct plate? (y/n)')
    
        if correct == 'y':
        
            new_header = fits.Header()
            
            try:
                new_header['Rcd'] = (t1['Rcd'], 'Record number in finding aid')
            except: 
                new_header['Rcd'] = ('', 'Record number in finding aid')

            try:
                new_header['Series'] = (t1['Series'], 'Glass plate series')
            except: 
                new_header['Series'] = ('', 'Glass plate series')

            try:
                new_header['PlateNum'] = (t1['Plate no.'], 'Plate number')
            except: 
                new_header['PlateNum'] = ('', 'Plate number')

            try:
                new_header['DATE-OBS'] = (t1['Date'], 'Date of observation from logbook')
            except:
                new_header['DATE-OBS'] = ('', 'Date of observation from logbook')
            
            try:
                new_header['Object'] = (t1['Object'], 'Object observered')
            except: 
                new_header['Object'] = ('', 'Object observered')
            
            
            try:
                new_header['RA'] = (t1['RA'], 'Right Ascension from logbook')
            except: 
                new_header['RA'] = ('', 'Right Ascension from logbook')

            
            try:
                new_header['Dec'] = (t1['Dec'], 'Declination from logbook')
            except: 
                new_header['Dec'] = ('', 'Declination from logbook')
            
            try:
                new_header['Exp'] = (t1['Exp (min)'], 'Exposure time (minutes)')
            except:
                new_header['Exp'] = ('', 'Exposure time (minutes)')
        
            try: 
                new_header['TIME-OBS'] = (t1['Start'], 'Exposure start time')
            except:
                new_header['TIME-OBS'] = ('', 'Exposure start time')
        
            try: 
                new_header['TIME-MID'] = (t1['Mid'], 'Exposure middle time')
            except:
                new_header['TIME-MID'] = ('', 'Exposure middle time')
        
            try:
                new_header['TIME-END'] = (t1['End'], 'Exposure end time')
            except:
                new_header['TIME-END'] = ('', 'Exposure end time')
            
            try:
                new_header['TIMESYS'] = (t1['Time type'], 'Timekeeping system/Time zone used')
            except:
                new_header['TIMESYS'] = ('', 'Timekeeping system/Time zone used')
                
            try: 
                new_header['Emulsion'] = (t1['Emulsion'], 'Emulsion of Glass Plate')
            except:
                new_header['Emulsion'] = ('', 'Emulsion of Glass Plate')
        
            
            try: 
                new_header['Filter'] = (t1['Filter'], 'Filter used during glass plate exposure')
            except:
                new_header['Filter'] = ('', 'Filter used during glass plate exposure')
                
    
            try:
                new_header['PlateSiz'] = (t1['Plate size (inches)'], 'Size of glass plate (inches)')
            except:
                new_header['PlateSiz'] = ('', 'Size of glass plate (inches)')
            
            try: 
                new_header['Observer'] = (t1['Observer'], 'Observer from logbook')
            except: 
                new_header['Observer'] = ('', 'Observer from logbook')

            
            try:
                new_header['Location'] = (t1['Location'], 'Known current location of glass plate')
            except:
                new_header['Location'] = ('', 'Known current location of glass plate')
            
            try:
                new_header['Notes'] = (t1['Notes'], 'Notes from logbook')
            except: 
                new_header['Notes'] = ('', 'Notes from logbook')

            correct_plate = True
        
    return new_header
            




def mask_artifacts(file, limit, new_value, makefile = True):

    with fits.open(file) as hdu:
        data = hdu[0].data
        header = hdu[0].header
        
    
    new_data = data.copy()
    row, column = np.shape(data)
    cut = 3
    
    print(np.min(data))
    
    for r in range(row):
        for c in range(column):
            if data[r,c] < limit:
                for r2 in range(r - cut, r + cut+1):
                    if r2 < 0 or r2 > (row - 1):
                        continue
                    for c2 in range(c - cut, c + cut+1): 
                        if c2 < 0 or c2 > (column - 1):
                            continue
                        if data[r2,c2] >= limit or new_data[r2,c2] == new_value:
                            continue
                        else: 
                            new_data[r2,c2] = new_value

    plt.figure(figsize = (10,5))
    plt.hist(new_data.flatten(), bins = 10000, log = True)
    plt.show()
    
    
    print(min(new_data.flatten()))

    newfilepath = file[:-5] + '_CLEAN.fits'
    print(newfilepath)
    
    if makefile == True:
        fits.writeto(newfilepath, new_data, header, overwrite = True)
    return newfilepath

In [ ]:
# enter path of directory that contains all of the tif files that you want to convert to fits files
    # there should be no subdirectories between this path and the image it self
    # e.g.) path = 'Plate_Scans/R-1560/'

#path = ''

####### CHANGE THE PATH #######

path = '/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-4017/'

tif_filelist = glob(path + '*.tif')

print(tif_filelist)

In [ ]:
# This will loop through the list of tif files that you created, convert them into fits files.
# Input your finding aid information to add it to the header.

fits_filelist = []
for i,file in enumerate(tif_filelist):
    header_input = input(f'\nDo you have finding aid information associated with plate {os.path.basename(file)}? (y/n)')
    if header_input == 'y':
        newfname = tif2fits_RGB(file, write_header = True)
    elif header_input == 'n':
        newfname = tif2fits_RGB(file, write_header = False)
        
    fits_filelist.append(newfname)
    

if len(fits_filelist) == 1:
    print('\n\nDONE! 1 tif file was converted to a fits file! This is the file that will be analyzed.')
    selected_file = newfname
elif len(fits_filelist) == 0:
    print('Uh oh! 0 tif files were found! Confirm you provided the correct path!')
    selected_file = 0
else: 
    print('\n\nDONE! ' + str(len(fits_filelist)) + ' tif files were converted to fits files!')

In [ ]:
# select which fits file you would like to further analyze (this cell can be rerun in order to analyze different files)
# You typically want to analyze the inner scan.

if len(fits_filelist) > 1:  
    for f, file in enumerate(fits_filelist):
        print('index: ' + str(f) + '  \n\t' + os.path.basename(file))

    file_selection = int(input('\n\n\nBased on the above output, what is the index of the file you would like to analyze?'))
    if file_selection > len(fits_filelist):
        print ('\nUh oh! This index is out of range. Please try again.')
        selected_file = 0
    else: 
        selected_file = fits_filelist[file_selection]
        print('\nYou\'ve selected file ' + str(selected_file) + '\nThis is the file that will be analyzed.')

In [ ]:
# display image

###### change if necessary ######

selected_file = '/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-4017/YK-24-R-4017-Epson10000-1600-inner-001_WCS.fits'
with fits.open(selected_file) as hdu:
    data = hdu[0].data
    header = hdu[0].header
    norm = ImageNormalize(stretch=SqrtStretch())   
    plt.figure()
    plt.imshow(data, origin='lower', cmap='Greys_r', norm=norm, interpolation='nearest')
    plt.show()

#### Step Three
Clean the image.

In [ ]:
# view histogram of image

# if you need to call the file again for some reason
#selected_file = '/raid6/users/rantoine/R_Aql/Plates/Good/R-494/YK-24-R-494-Epson10000-1600-inner-001_WCS.fits'
%matplotlib ipympl

with fits.open(selected_file) as hdu:
    data = hdu[0].data
    header = hdu[0].header
        
    plt.figure(figsize = (10,5))
    plt.hist(data.flatten(), bins = 10000, log = True)
    plt.show()

In [ ]:
#call function to mask artifacts and create a "clean" file
fits_filelist = glob(selected_file)

file1 = fits_filelist[0]
print(file1)

####### set your minimum based on above histogram #######

clean_file = mask_artifacts(file1, limit = 10000, new_value = 10000, makefile = True)

In [ ]:
plt.close()

#### Step Four
Solve with astrometry.net. Now, note that astrometry.net has been finnicky as of late, so retrieve WCS files in whatever way you see fit and skip this step if needed.

In [ ]:
# do astronometry.net solve for selected fits file


scan = clean_file
ast = AstrometryNet()

####### enter your nova.astrometry.net api key #######

#ast.api_key = ''

ast.api_key = ''

search_type = input('Provide RA and Dec? (header/input/none)')

if search_type == 'input':
    
    # if yes, you will be asked the ra, dec, and platescale radius to give astrometry a rough estimate of what field to search for
    
    ra = input('\nRA in hh:mm:ss ? (round to nearest integers and remember to put leading zeroes)')
    ra_split = [int(x) for x in ra.split(':')]
    ra_deg = ( ra_split[0] + (ra_split[1]/60) +(ra_split[2]/3600) )* 15
    print('RA in degrees: ' + str(ra_deg))
    
    
    dec = input('\nDec in ±ºº:\'\':\"\" ? (put + or - sign, round to nearest integers, and remember leading zeroes)')
    dec_sign = int( dec.split(':')[0][0] + '1')
    dec_split = [int(x) for x in dec.split(':')]
    dec_deg = dec_sign * ( abs(dec_split[0]) + (dec_split[1]/60) +  (dec_split[2]/3600) )
    print('Dec in degrees: ' + str(dec_deg))
    
    
    rad = int(input('\nRadius in degrees?'))
    print('\n')
    wcs_header = ast.solve_from_image(scan, force_image_upload=True, 
                                      center_ra = ra_deg, center_dec = dec_deg, radius = rad , 
                                      crpix_center=True, allow_commercial_use = 'n')

    data = fits.getdata(scan)
    newfile =  scan[:-11] + '_WCS.fits'
    fits.writeto(newfile, data, header = wcs_header, overwrite=True)
    os.chmod(newfile, 0o777)
    print(newfile + ' has been saved! \n')
    
elif search_type == 'header':
    data = fits.getdata(scan)
    header = fits.getheader(scan)
    ra_deg = header['RA']
    dec_deg = header['DEC']

    # radius is set at a default of 20º
    wcs_header = ast.solve_from_image(scan, force_image_upload=True, 
                                      center_ra = ra_deg, center_dec = dec_deg, radius = 20 , 
                                      crpix_center=True, allow_commercial_use = 'n')
    header.extend(wcs_header)
    newfile =  scan[:-11] + '_WCS.fits'
    fits.writeto(newfile, data, header = header, overwrite=True)
    os.chmod(newfile, 0o777)
    print(newfile + ' has been saved! \n')
    
elif search_type == 'none':
    # if no, astrometry will do a blind search which make take longer to get a solution
    wcs_header = ast.solve_from_image(scan, force_image_upload=True,  
                                      crpix_center=True, allow_commercial_use = 'n')
    data = fits.getdata(scan)
    newfile =  scan[:-11] + '_WCS.fits'
    fits.writeto(newfile, data, header = wcs_header, overwrite=True)
    os.chmod(newfile, 0o777)
    print(newfile + ' has been saved! \n')
    
else: 
    print('Invalid entry')

#### Step Five
Now that you have your WCS file, you can start subtracting the background. Skip to here if you already have a clean WCS file.

In [ ]:
files = glob(path + '*WCS.fits')
#files = glob(path + '*image.fits')
#files = glob(path + '*CLEAN.new')
#files = glob(path + 'L-76-006.fits')
#files = glob(path + '*c.new')
file = files[0]
filename = os.path.basename(file)
print(filename)

In [ ]:
# Defining functions and center mask. No need to change anything unless you need a center mask.

# if no center mask:

c_mask = None

# various analysis functions


# def gaia() runs a querey to pull sources within our field

def gaia(header):
    wcs_header = WCS(header)
    bottom_coord = wcs_header.pixel_to_world(0, 0)
    top_coord = wcs_header.pixel_to_world(header['NAXIS1'], header['NAXIS2'])
    full_coord_ra = np.abs(top_coord.ra.deg-bottom_coord.ra.deg)
    full_coord_dec = np.abs(top_coord.dec.deg-bottom_coord.dec.deg)
    ra_center = (top_coord.ra.deg + bottom_coord.ra.deg) / 2
    dec_center = (top_coord.dec.deg + bottom_coord.dec.deg) / 2
    job = Gaia.launch_job("SELECT TOP 300000 "
                    "source_id,ra,dec,parallax,parallax_error,pm,pmra,pmra_error,pmdec,pmdec_error,"
                    "phot_g_mean_mag, phot_g_mean_flux, phot_bp_mean_mag,phot_bp_mean_flux,phot_rp_mean_mag,"
                    "phot_rp_mean_flux,bp_rp, phot_variable_flag, classprob_dsc_combmod_galaxy"
                    " from gaiadr3.gaia_source"
                    " WHERE CONTAINS(POINT('ICRS',ra,dec),BOX('ICRS',{0},{1},{2},{3}))=1 AND"
                    "((phot_bp_mean_mag + 0.9*bp_rp) <= 19.0)".format(ra_center,
                                                                      dec_center,
                                                                    full_coord_ra, full_coord_dec))
    gaia_res = job.get_results()
    gaia_coord = SkyCoord(gaia_res['ra'], gaia_res['dec'], frame = 'icrs', unit = 'deg')
    
    return gaia_res, gaia_coord


 # def ap_phot2 detects and subtracts the image background through photutils and then extracts objects from image with SEP, calculating a mag

def ap_phot2(data,  sigma, msize, fsize, c_mask):
    
    sigma_clip = SigmaClip(sigma = sigma)
    bkg_estimator = MMMBackground()
    
    bkg = Background2D(data, box_size=(msize, msize), mask = c_mask, filter_size=(fsize, fsize), 
                       sigma_clip=sigma_clip, bkg_estimator=bkg_estimator)
    data_sub = data - bkg.background 

    objects = Table(sep.extract(data_sub, sigma, err = bkg.background_rms_median))
    
    ob_mask = np.where(objects['flag'] == 0)[0] #changed from !=8 to =0
    objects = objects[ob_mask]


    
    objects['flux2'], sum_err, flag = sep.sum_ellipse(data=data_sub, x=objects['x'], y=objects['y'], a=objects['a'], b=objects['b'], theta=objects['theta'],err=bkg.background_rms_median)
    
    objects['sep_mag'] = -2.5 * np.log10(objects['flux2']) 

    
    
    return data_sub, objects


# def match_table matches the sources found within our image to the sources found in the gaia querey, returning a table with these matches results

def match_table(object_table, wcs, gaia_table, gaia_coordinates):
    print(len(object_table), len(gaia_table))
    pcor = wcs.pixel_to_world(object_table['x'], object_table['y'])
    object_table['ra_p'], object_table['dec_p'] = pcor.ra.deg, pcor.dec.deg
    index, d2d, ____ = pcor.match_to_catalog_sky(gaia_coordinates)
    mtab = hstack([gaia_table[index], Table(object_table)])
    _, unique_ind = np.unique(mtab['source_id'], return_index=True)
    mtab = mtab[unique_ind]
    
    mtab['ang_dist'] = np.abs(np.sqrt(mtab['ra']**2+mtab['dec']**2)
                              -np.sqrt(mtab['ra_p']**2+mtab['dec_p']**2))
    mtab['dec_res'] = (mtab['dec'] -  mtab['dec_p'])*3600
    mtab['ra_res'] = (mtab['ra'] -  mtab['ra_p'])*3600*0.9114

    
    
            
    return mtab


# def make_plots() makes plots displaying differences between our scan sources and gaia sources

def make_plots(tablenum, sigma):
    pg_list = list(tablenum['pg'])
    mag_list = list(tablenum['sep_mag'])
    newpglist = []
    newmaglist = []
    for i in range(len(pg_list)):
        newpglist.append(round(pg_list[i],2))
        newmaglist.append(round(mag_list[i],2))
    
    diff = spmode(newpglist)[0] - spmode(newmaglist)[0]
    
    diff_factor = diff
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
    fig.set_figheight(6)
    fig.set_figwidth(24)
    
    ax1.set_title('Sigma: %d, %d stars found! '%(sigma, len(tablenum)))
    ax1.scatter(tablenum['pg'],tablenum['sep_mag'] + diff_factor, s = 3, alpha = 0.2)
    ax1.plot(np.arange(min(tablenum['pg']),max(tablenum['pg'])+2),np.arange(min(tablenum['pg']),max(tablenum['pg'])+2), color = 'r', ls = '--', alpha = 0.5)
    ax1.scatter(spmode(newpglist)[0], spmode(newmaglist)[0]+ diff_factor)    
    ax1.set_xlabel('Phot Mag Approx from Gaia')
    ax1.set_ylabel('Phot Mag Approx from SEP')
    
    ax2.scatter(tablenum['pg'],tablenum['pg'] - tablenum['sep_mag']- diff_factor, s = 3, alpha = 0.2)
    ax2.set_title('Sigma: %d, %d stars found! '%(sigma, len(tablenum)))
    ax2.hlines(0,min(tablenum['pg']),max(tablenum['pg'])+2, color='r', ls = '--', alpha = 0.5)
    ax2.set_xlabel('Phot Mag Approx from Gaia')
    ax2.set_ylabel('Residual of Phot Mag Approx from SEP and Gaia')
    
    ax3.scatter(tablenum['x'],tablenum['pg'] - tablenum['sep_mag']- diff_factor, s = 3, alpha = 0.2)
    ax3.set_title('Sigma: %d, %d stars found! '%(sigma, len(tablenum)))
    ax3.set_xlabel('X Coordinate')
    ax3.set_ylabel('Residual of Phot Mag Approx from SEP and Gaia')

    plt.show()

    print('Phot Mag Approx from Gaia: ' + str(mean(tablenum['pg'])))
    print('\nPhot Mag Approx from SEP: ' + str(mean(tablenum['sep_mag']) + diff_factor))
    print('\nResidual of Phot Mag Approx from SEP and Gaia: ' + str(mean(tablenum['pg'] - (tablenum['sep_mag'])- diff_factor) ))
    print('STD Residual: ' + str(std(tablenum['pg'] - (tablenum['sep_mag'])) ))


# def make_bkg_pdf() creates a pdf with plots of detected background and rms for easy review

def make_bkg_pdf(path, filename, pdf_name, svalue, c_mask, std_lim, weight_lim = 0.6):
    # lists that contain ranges of mesh and filter sizes we want to test 
    # you may want to change these values, if you want to test more mesh and filter sizes
    
    # define chosen parameters for background detection and plots 
    norm = ImageNormalize(stretch=SqrtStretch())
    bkg_estimator = MMMBackground()
    
    with fits.open(path+filename) as hdu:
        data = hdu[0].data.astype(np.float32)
        header = hdu[0].header

    mlist = list(np.linspace(   round(data.shape[0]/50),    round(data.shape[0]/10),     5, dtype=int))
    flist = list(np.arange(15,26,2))

    with PdfPages(path + pdf_name) as pdf:
        
        sigma_clip = SigmaClip(sigma = svalue, maxiters = None )

        # loop through mesh sizes 
        for m, msize in enumerate(mlist): 
            print('\nMesh size: ' + str(msize))
            
            # loop through filter sizes 
            for k, fsize in enumerate(flist):
                print('Filter size: ' + str(fsize))

                # detect background through photutils
                bkg = Background2D(data, (msize, msize), filter_size=(fsize, fsize), sigma_clip=None, bkg_estimator=bkg_estimator, mask = c_mask)

                # certain parameters return rms plots with odd patterns. limiting the standard deviation within the background rms 
                # will minimize the amount of these plots you have to parse through

                if std(bkg.background_rms) < 0.0002:
                    print('Sigma %d, Mesh %d, Filter %d skipped. Has rms std of %.4f'%(svalue, msize, fsize, std(bkg.background_rms)))
                    continue
                
                elif std(bkg.background_rms) < std_lim: 
                    # subtract background from data 
                    new_phot_data = data - bkg.background
                    
                    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
                    fig.set_figheight(8)
                    fig.set_figwidth(15)
                    fig.suptitle('\nSigma %d,   Mesh %d,   Filter %d\n  Min: %.4f'%(svalue, msize, fsize, np.min(new_phot_data)))
                    #fig.suptitle('\nSigma %d,   Mesh %d,   Filter %d\n '%(svalue, msize, fsize))
                    
                    p1 = ax1.imshow(bkg.background, origin='lower', cmap='Greys_r', interpolation='nearest')
                    plt.colorbar(p1, ax = ax1, shrink=0.4)
                    ax1.set_title('Bkg')
                                                                
                    p2 = ax2.imshow(bkg.background_rms, origin='lower', cmap='Greys_r', interpolation='nearest')
                    plt.colorbar(p2, ax = ax2, shrink=0.4)
                    ax2.set_title('RMS\n Med: %.4f   STD: %.4f'%(bkg.background_rms_median, std(bkg.background_rms)))
                    
                    ax3.imshow(new_phot_data, origin='lower', cmap='Greys_r', norm=norm, interpolation='nearest')
                    bkg.plot_meshes(outlines=True, marker='.', color='cyan', alpha=0.3)

                    pdf.savefig()
                    plt.close()
                     
                else: 
                    print('Sigma %d, Mesh %d, Filter %d has %.4f'%(svalue, msize, fsize, std(bkg.background_rms)))

    print('PDF is complete!')


# def find_sigma() displays plots at varying sigma values, so you can select which sigma has the best "shape"

def find_sigma(path, filename, c_mask, mbox, test_fsize = 21, test_alpha = 0.9):
    slist = list(np.arange(3,16,2))
    
    with fits.open(path+filename) as hdu:
        data = hdu[0].data.astype(np.float32)
        header = hdu[0].header

    wcs = WCS(header)

    # provide middle ground alpha just to test sigma
    gtab, gcor = gaia(header)

    gtab['pg'] = gtab['phot_bp_mean_mag']+ test_alpha * gtab['bp_rp']
    gtab['pg_flux'] = gtab['phot_bp_mean_flux']+ test_alpha *(gtab['phot_bp_mean_flux']
                                                             /gtab['phot_rp_mean_flux'])
    
 

    for s, svalue in enumerate(slist):
        print('\n\nSigma value: ' + str(svalue))

        #objects from scan 
        # provide middle group mask and filter size just to test sigma 
        test_msize = round(data.shape[0]/mbox)
        
        data_sub, otab = ap_phot2(data, svalue, test_msize, test_fsize, c_mask)
    
        #match of gaia and s
        mtab = match_table(otab,wcs,gtab,gcor)

        print(str(len(mtab))+ ' stars found!')

        make_plots(mtab, svalue)



# def alpha_finder() finds the alpha value with the smallest differences between gaia mag and sep mag 

def alpha_finder(data_sub, wcs, otab, gtab, gcor):

    a_list = list(np.round(np.arange(-0.5,1.5,0.1), 2))
    avglist = []
    avglist2 = []

    for a, alpha_val in enumerate(a_list):
        #print('Alpha Value: ' + str(alpha_val))
        alpha = alpha_val
        
        # photographic magnitude approximation

    
        
        gtab['pg'] = gtab['phot_bp_mean_mag']+ alpha * gtab['bp_rp']
        gtab['pg_flux'] = gtab['phot_bp_mean_flux']+ alpha *(gtab['phot_bp_mean_flux']
                                                                 /gtab['phot_rp_mean_flux'])
        
        #match of gaia and s
        mtab = match_table(otab,wcs,gtab,gcor)

        scal = np.max(mtab['flux2'])/np.max(data_sub) 
        popt, ___ = curve_fit(sq, mtab['flux2']/scal, mtab['pg_g_flux']/np.pi)    
        ndata = sq(data_sub,*popt)    
        
        avglist.append(mean(mtab['pg_g'] - ((mtab['sep_mag']))))
        pg_list = list(mtab['pg_g'])
        mag_list = list(mtab['sep_mag'])
        newpglist = []
        newmaglist = []
        for i in range(len(pg_list)):
            newpglist.append(np.round(pg_list[i],3))
            newmaglist.append(np.round(mag_list[i],3))

        diff = spmode(newpglist)[0] - spmode(newmaglist)[0]
        avglist2.append(mean(mtab['pg_g'] - ((mtab['sep_mag']+diff))))
    
    avgmin = min(np.abs(avglist))
    index = np.where(np.abs(avglist) == avgmin)[0][0]
    alpha1 = a_list[index]
    
    avgmin2 = min(np.abs(avglist2))
    index2 = np.where(np.abs(avglist2) == avgmin2)[0][0]
    alpha2 = a_list[index2]

    print('Alpha 1: %.1f and Alpha 2: %.1f'%(alpha1, alpha2))

    return alpha1, alpha2


# def analysis 1 and 2 call all of our functions to create a table and alter our data 
def analysis_pt1(path, filename, sigma, msize, fsize, c_mask):
    with fits.open(path + filename) as hdu:
        data = hdu[0].data.astype(np.float32)
        header = hdu[0].header
        #header['EXPTIME'] = (3600.0, 'Exposure time in seconds')
        #header['TSCOPE'] = 'Y41'
        #header['OBSDATE'] = ('1979-07-02', 'YYYY-MM-DD')
        #header['EMULSION'] = '103a-E'
        #header['PLTSIZE'] = ('3.25x4.25', 'Plate size in inches')
        
    wcs = WCS(header)
    #gaia querey results 
    gtab, gcor = gaia(header)
    
    #objects from scan 
    data_sub, otab = ap_phot2(data,  sigma, msize, fsize, c_mask)
    alpha1, alpha2 = alpha_finder(data_sub, wcs, otab, gtab, gcor)
    print('Using alpha: ' + str(alpha2))
    # photographic magnitude approximation
        
   

    gtab['pg'] = gtab['phot_bp_mean_mag']+ alpha2 * gtab['bp_rp']
    gtab['pg_flux'] = gtab['phot_bp_mean_flux']+ alpha2 *(gtab['phot_bp_mean_flux']
                                                             /gtab['phot_rp_mean_flux'])
    
    #match of gaia and scan
    mtab = match_table(otab,wcs,gtab,gcor)
    
    scal = np.max(mtab['flux2'])/np.max(data_sub) 
    
    popt, ___ = curve_fit(sq, mtab['flux2']/scal, mtab['pg_flux']/np.pi) 
    ndata = sq(data_sub,*popt) 
    return mtab, ndata, header, data_sub, gtab, gcor


def analysis_pt2(ndata, sigma, msize, fsize, c_mask, gtab, gcor):
    wcs = WCS(header)
    data_sub, otab = ap_phot2(ndata,  sigma, msize, fsize, c_mask)
    mtab = match_table(otab, wcs, gtab, gcor)
    return mtab, data_sub

In [ ]:
plt.close()
# Decide the best sigma by looking for the "backwards california" to lie along the red diagonal with few outliers.
# display plots at different sigma values (with set alpha, mesh, and filter)
%matplotlib inline

#print(filename) # this should be your saved _WCS.fits file

     #if something went wrong in your astrometry.net solve, change the filename variable below
#filename = 'new-image.fits'


mbox_num = 50    # make sure  5 ≤ mbox_num ≤ 100

    # mbox_num defines how many columns of boxes you want your background mesh to contain
    # if your analysis isnt working, consider changing this number


find_sigma(path, filename, c_mask, mbox = mbox_num)

# additional notes: 
    # msize indicates how many pixels wide each box of the background mesh is
    # our test value is taking the shape of the data and dividing it by our mbox_num variable

In [ ]:
# input your best sigma and a std limit, this creates a pdf file of detected backgrounds
plt.close()
sigma = int(input('What is the best sigma value? '))

# Make a rough guess on what the typical standard deviation is going to be
# If you run the code and everything is being rejected, interrupt the kernel and raise the limit 
slim = int(input('What is your standard deviation limit? ')) #start with std = 100

make_bkg_pdf(path = path, filename = filename, pdf_name = 'bkg_plots.pdf', svalue = sigma, c_mask = c_mask, std_lim = slim)

In [ ]:
# Open the backgrounds.pdf and look for a dark and smooth rms.

# input your best sigma, mesh size, and filter size 

sigma = float(input('What is the best sigma value? '))
mesh_size = int(input('What is your mesh size? '))
filter_size =int(input('What is your filter size? '))

In [ ]:
 with fits.open(path + filename) as hdu:
    data = hdu[0].data
    header = hdu[0].header
    
data_sub, objects = ap_phot2(data,  sigma, mesh_size, filter_size, c_mask)

# view histogram of image
%matplotlib inline

print(f'Count minimum: {min(data_sub.flatten())} \nCount maximum: {max(data_sub.flatten())} \nCount mode: {statistics.mode(data_sub.flatten())}')

        
plt.figure(figsize = (10,5))
plt.hist(data_sub.flatten(), bins = 10000, log = True)
plt.show()

#### Step Six
Run the plate analysis and save your results.

In [ ]:
# runs part one of analysis

%matplotlib ipympl
mtab1, ndata, header, data_sub, gtab, gcor= analysis_pt1(path, filename, sigma, mesh_size, filter_size, c_mask)
make_plots(mtab1, sigma)

In [ ]:
# runs part two of analysis

# need to figure out error, but part one is fine for now on its own

%matplotlib ipympl
mtab2, ndata2 = analysis_pt2(ndata, sigma, mesh_size, filter_size, c_mask, gtab, gcor)
make_plots(mtab2, sigma)

In [ ]:
# adds sigma, mesh, and filter sizes to header of background image
# also adds alpha value to header of final analyzed images and saves match table

header['SIGMA'] = (sigma, 'Sigma value used during background subtraction and aperture photometry')
header['BKG_M'] = (mesh_size, 'Mesh size used during background subtraction')
header['BKG_F'] = (filter_size, 'Filter size used during background subtraction')

fits.writeto(files[0][:-8]+'g_bkg_sub.fits' , data_sub, header = header, overwrite=True)




# save ndata as a fits file

header['ALPHA'] = 1.0

fits.writeto(files[0][:-8]+'g_ANALYZED1.fits' , ndata, header = header, overwrite=True)

mtab1.write(files[0][:-8]+'g_TABLE1.csv' , format = 'ascii.csv', overwrite=True)

#fits.writeto(files[0][:-8]+'ANALYZED2.fits' , ndata2, header = header, overwrite=True)
#mtab2.write(files[0][:-8]+'TABLE2.csv' , format = 'ascii.csv', overwrite=True)

In [ ]:
# display plots at different sigma values (with set alpha, mesh, and filter)
%matplotlib inline

#print(filename) # this should be your saved _WCS.fits file

    # if something went wrong in your astrometry.net solve, change the filename variable below
#filename = 'YK-24-R-80-Epson10000-1600-inner-001_WCS.fits'


mbox = 50    # make sure  5 ≤ mbox_num ≤ 100

    # mbox_num defines how many columns of boxes you want your background mesh to contain
    # if your analysis isnt working, consider changing this number


fsize = 15
test_alpha = 0.9


with fits.open(path+filename) as hdu:
    data = hdu[0].data.astype(np.float32)
    header = hdu[0].header

wcs = WCS(header)


# provide middle ground alpha just to test sigma
gtab, gcor = gaia(header)

    
'''gtab['pg_g'] = gtab['phot_g_mean_mag']+ test_alpha * gtab['bp_rp']
gtab['pg_g_flux'] = gtab['phot_g_mean_flux']+ test_alpha *(gtab['phot_bp_mean_flux']
                                                             /gtab['phot_rp_mean_flux'])'''

gtab['pg'] = gtab['phot_bp_mean_mag']+ test_alpha * gtab['bp_rp']
gtab['pg_flux'] = gtab['phot_bp_mean_flux']+ test_alpha *(gtab['phot_bp_mean_flux']
                                                         /gtab['phot_rp_mean_flux'])

####### CHANGE SIGMA #######

sigma = 13
print('\n\nSigma value: ' + str(sigma))

#objects from scan 
# provide middle group mask and filter size just to test sigma 
msize = round(data.shape[0]/mbox)


sigma_clip = SigmaClip(sigma = sigma)
bkg_estimator = MMMBackground()

bkg = Background2D(data, box_size=(msize, msize), mask = c_mask, filter_size=(fsize, fsize), 
                   sigma_clip=sigma_clip, bkg_estimator=bkg_estimator)
data_sub = data - bkg.background 

objects = Table(sep.extract(data_sub, sigma, err = bkg.background_rms_median))
print(len(objects))

ob_mask = np.where(objects['flag'] != 8)[0]
objects = objects[ob_mask]


#objects['flux2'], sum_err, flag = sep.sum_circle(data=data_sub, x=objects['x'], y=objects['y'], r = ((objects['xmax'] - objects['xmin'])/2), err=bkg.background_rms_median)

objects['flux2'], sum_err, flag = sep.sum_ellipse(data=data_sub, x=objects['x'], y=objects['y'], a=objects['a'], b=objects['b'], theta=objects['theta'],err=bkg.background_rms_median)

objects['sep_mag'] = -2.5 * np.log10(objects['flux2'])


#data_sub, otab = ap_phot2(data, svalue, test_msize, test_fsize, c_mask)


#match of gaia and s
mtab = match_table(objects,wcs,gtab,gcor)

print(str(len(mtab))+ ' stars found!')

make_plots(mtab, sigma)

In [ ]:
df = pd.read_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-80/YK-24-R-80-Epson10000-1600-inner-001_TABLE1.csv', dtype={"source_id": str})
print(df['source_id'])

target_df = df[df['source_id'] == '4307450193249412352']
standard_df = df[df['source_id'] == '4307372089263785216']

print(target_df)
print(standard_df)
standard_df.to_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-80/standard_TABLE1.csv')

In [ ]:
####### CHANGE THE FILE NAME #######

df = pd.read_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-495/YK-24-R-495-Epson10000-1600-inner-001_CTABLE1.csv', dtype={"source_id": str})

target_df = df[df['source_id'] == '4307450193249412352']
pd.set_option('display.max_columns', None)
print(target_df)

####### CHANGE THE FILE NAME #######

target_df.to_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-495/R_Aql_TABLE2.csv', index=False)

In [ ]:
# display image
%matplotlib widget
df = pd.read_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-4017/YK-24-R-4017-Epson10000-1600-inner-001_g_TABLE1.csv', dtype={"source_id": str})
path = '/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-4017/'
filename = 'YK-24-R-4017-Epson10000-1600-inner-001_WCS.fits'
with fits.open(path + filename) as hdu:
    data = hdu[0].data
    header = hdu[0].header
    norm = ImageNormalize(stretch=SqrtStretch())   
    plt.figure()
    plt.imshow(data, origin='lower', cmap='Greys_r', norm=norm, interpolation='nearest')
    plt.scatter(df['x'], df['y'], s =10
               )
    plt.show()

In [ ]:
with fits.open(path + filename) as hdu:
    data = hdu[0].data
    header = hdu[0].header
    norm = ImageNormalize(stretch=SqrtStretch())   
    plt.figure()
    plt.imshow(data, origin='lower', cmap='Greys_r', norm=norm, interpolation='nearest')
    plt.scatter(objects['x'], objects['y'], s =0.08)
    plt.show()

In [ ]:
with fits.open(path + filename) as hdu:
    data = hdu[0].data
    header = hdu[0].header
    norm = ImageNormalize(stretch=SqrtStretch())   
    plt.figure()
    plt.imshow(data, origin='lower', cmap='Greys_r', norm=norm, interpolation='nearest')
    plt.scatter(mtab['x'], mtab['y'], s =0.08)
    plt.show()

In [ ]:
#df = pd.read_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/6B-1105/YK-6-6B-1105-Epson10000-1600-inner-001_CTABLE1.csv', dtype={"source_id": str})
#print(df)
err = 10

df = objects.to_pandas()
tx, ty = (5234.6, 6199.6)
match = df[(df['x'].sub(tx).abs() <= err) &
        (df['y'].sub(ty).abs() <= err)]
print(match)

#### Part IDK WHATS HAPPENING

In [ ]:
#gonna try lookin at what i got thus far
path = '/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/'

mag_files = sorted(glob(path + '*/*R_Aql_TABLE2.csv'))
pg_mag = []
g_mag = []
bp_mag = []
rp_mag = []
bp_rp = []
for filepath in mag_files:
    
    df = pd.read_csv(filepath)
    pg = df['pg']
    g = df['phot_g_mean_mag']
    bp = df['phot_bp_mean_mag']
    rp = df['phot_rp_mean_mag']
    bprp= df['bp_rp']
    pg_mag.append(pg)
    g_mag.append(g)
    bp_mag.append(bp)
    rp_mag.append(rp)
    bp_rp.append(bprp)
print(pg_mag)
print(g_mag)
print(bp_mag)
print(rp_mag)
print(bp_rp)

s_mag_files = sorted(glob(path + '*/standard_TABLE1.csv'))
s_pg_mag = []
s_g_mag = []
s_bp_mag = []
s_rp_mag = []
s_bp_rp = []
for filepath in s_mag_files:
    
    df = pd.read_csv(filepath)
    pg = df['pg']
    g = df['phot_g_mean_mag']
    bp = df['phot_bp_mean_mag']
    rp = df['phot_rp_mean_mag']
    bprp= df['bp_rp']
    s_pg_mag.append(pg)
    s_g_mag.append(g)
    s_bp_mag.append(bp)
    s_rp_mag.append(rp)
    s_bp_rp.append(bprp)
print(s_pg_mag)
print(s_g_mag)
print(s_bp_mag)
print(s_rp_mag)
print(s_bp_rp)

#df = pd.DataFrame({'Date': pd.to_datetime(dates), 'Avg_Flux': avg_flux})

In [ ]:
frames = []
for filepath in mag_files:
    df = pd.read_csv(filepath)
    df['source_file'] = filepath  # keep track of which plate/epoch it came from
    frames.append(df[['pg', 'phot_g_mean_mag', 'phot_bp_mean_mag',
                       'phot_rp_mean_mag', 'bp_rp', 'source_file']])

combined = pd.concat(frames, ignore_index=True)

pg = combined['pg']
g_mag = combined['phot_g_mean_mag']
bp_mag = combined['phot_bp_mean_mag']
rp_mag = combined['phot_rp_mean_mag']
bp_rp = combined['bp_rp']
date = combined['Date']

print(combined)

s_frames = []
for filepath in s_mag_files:
    df = pd.read_csv(filepath)
    df['source_file'] = filepath  # keep track of which plate/epoch it came from
    s_frames.append(df[['pg', 'phot_g_mean_mag', 'phot_bp_mean_mag',
                       'phot_rp_mean_mag', 'bp_rp', 'source_file']])

s_combined = pd.concat(s_frames, ignore_index=True)

s_pg = s_combined['pg']
s_g_mag = s_combined['phot_g_mean_mag']
s_bp_mag = s_combined['phot_bp_mean_mag']
s_rp_mag = s_combined['phot_rp_mean_mag']
s_bp_rp = s_combined['bp_rp']
s_date = s_combined['Date']

print(s_combined)

In [ ]:
pd.set_option('display.max_colwidth', None)
combined['source_file']

In [ ]:
df = pd.read_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-4017/R_Aql_TABLE2.csv')
date = datetime(1919, 9, 24) # year, month, date

df['Date'] = date.strftime('%Y-%m-%d')
df.to_csv('/raid6/users/rantoine/Mira_Notebooks/Part_Two/Plates/Good/Analyzed/R-4017/R_Aql_TABLE2.csv', index=False)

In [ ]:
frames = []
for filepath in mag_files:
    df = pd.read_csv(filepath)
    df['source_file'] = filepath  # keep track of which plate/epoch it came from
    frames.append(df[['pg', 'phot_g_mean_mag', 'phot_bp_mean_mag',
                       'phot_rp_mean_mag', 'bp_rp', 'source_file', 'Date']])

combined = pd.concat(frames, ignore_index=True)

pg_bp = combined['pg']
g_mag = combined['phot_g_mean_mag']
bp_mag = combined['phot_bp_mean_mag']
rp_mag = combined['phot_rp_mean_mag']
bp_rp = combined['bp_rp']
date = combined['Date']

print(combined)

s_frames = []
for filepath in s_mag_files:
    df = pd.read_csv(filepath)
    df['source_file'] = filepath  # keep track of which plate/epoch it came from
    s_frames.append(df[['pg', 'phot_g_mean_mag', 'phot_bp_mean_mag',
                       'phot_rp_mean_mag', 'bp_rp', 'source_file', 'Date']])

s_combined = pd.concat(s_frames, ignore_index=True)

s_pg_bp = s_combined['pg']
s_g_mag = s_combined['phot_g_mean_mag']
s_bp_mag = s_combined['phot_bp_mean_mag']
s_rp_mag = s_combined['phot_rp_mean_mag']
s_bp_rp = s_combined['bp_rp']
s_date = s_combined['Date']

print(combined)

In [ ]:
plt.close()


plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(combined['Date'], combined['pg'], color='red', s=15, label = 'R Aql')
ax.scatter(s_combined['Date'], s_combined['pg'], color = 'blue', s=15, label = 'Standard')
ax.set_xlabel('Observation')
ax.set_ylabel('apporx. photographic magnitude')
ax.invert_yaxis()
ax.set_title('Plate Light Curve of R Aql and Standard Star')


plt.legend()
fig.autofmt_xdate()
plt.savefig('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/something_is_wrong_light_curve.png')
plt.show()

In [ ]:
data = Table.read('/raid6/users/rantoine/R_Aql/Light_Curves/EPOCH_PHOTOMETRY-Gaia DR3 4307450193249412352.fits')
print(data.colnames)
time = data['bp_obs_time']
mag = data['bp_mag']

plt.figure(figsize=(10, 5))
plt.scatter(time, mag, color='teal', s=10)
plt.gca().invert_yaxis()
plt.xlabel('Time (Days)')
plt.ylabel('Gaia BP Magnitude')
plt.title(f'Gaia DR3 Light Curve for R Aql Source # {4307450193249412352}')
plt.grid(True, alpha=0.3)
plt.show()